In [1]:
"""
CNN-Transformer Hybrid Model for ECG Arrhythmia Classification
===============================================================
Fuses ECG spectrogram features (via CNN backbone) with patient metadata
(via MLP projection) and classifies arrhythmias via a Transformer encoder.

Input:
    spectrograms : (B, 1, H, W)  – single-channel time-frequency image
    metadata     : (B, M)        – continuous/encoded patient features

Output:
    logits       : (B, num_classes)
"""

'\nCNN-Transformer Hybrid Model for ECG Arrhythmia Classification\n===============================================================\nFuses ECG spectrogram features (via CNN backbone) with patient metadata\n(via MLP projection) and classifies arrhythmias via a Transformer encoder.\n\nInput:\n    spectrograms : (B, 1, H, W)  – single-channel time-frequency image\n    metadata     : (B, M)        – continuous/encoded patient features\n\nOutput:\n    logits       : (B, num_classes)\n'

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from typing import Optional


### CNN Backbone (EfficientNet-B0 with custom patch projection)

We decided to use EfficientNet-B0, a pretrained CNN that is smaller and more efficient than other pretrained models, as our CNN backbone. We implemented layer freezing, which prevents the layer gradients from updating, in order to prevent the model for overfitting the training data. Since EfficientNet-B0 was trained on photos, each layer represents a learned feature. The earlier stages represent low level features, such as edges and gradients, while the later layers represent more advanced features such as faces, animals, and other distinct objects. We freeze the lower levels because they will transfer well to spectrograph data. 

In [3]:
class CNNBackbone(nn.Module):
    """
    Extracts spatial feature maps from an ECG spectrogram, then flattens
    spatial positions into a sequence of patch tokens.

    Args:
        embed_dim   : Desired token dimension D.
        pretrained  : Whether to use ImageNet-pretrained EfficientNet weights.
        freeze_early: If True, freeze the first `freeze_layers` CNN stages
                      to preserve low-level features during fine-tuning.
        freeze_layers: Number of EfficientNet MBConv stages to freeze (0–7).
    """

    def __init__(
        self,
        embed_dim: int = 256,
        pretrained: bool = True,
        freeze_early: bool = True,
        freeze_layers: int = 4,
    ):
        super().__init__()
        weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        backbone = efficientnet_b0(weights=weights)

        # Remove the classifier and avgpool — we keep the feature map
        self.features = backbone.features  # (B, 1280, H', W')

        if freeze_early:
            # features[0] = stem, features[1..N] = MBConv stages
            for i in range(min(freeze_layers + 1, len(self.features))):
                for param in self.features[i].parameters():
                    param.requires_grad = False

        # Convert grayscale (1-channel) spectrogram to 3-channel for EfficientNet
        self.channel_expand = nn.Conv2d(1, 3, kernel_size=1, bias=False)

        # Project CNN feature map channels → embed_dim
        self.proj = nn.Sequential(
            nn.Conv2d(1280, embed_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(embed_dim),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, 1, H, W)
        Returns:
            tokens: (B, N, D)  where N = H' * W' (spatial patch count)
        """
        x = self.channel_expand(x)          # (B, 3, H, W)
        x = self.features(x)                # (B, 1280, H', W')
        x = self.proj(x)                    # (B, D, H', W')
        B, D, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)    # (B, N, D),  N = H'*W'
        return x

### Metadata Encoder

We needed to get our metadata (6 input features) into the same dimensional space as our CNN model, which produces 256 length 1-dimensional tensors. We deployed a simple MLP with one hidden layer using linear transformations in order to get our transformed output. 

In [4]:
class MetadataEncoder(nn.Module):
    """
    Projects a flat metadata vector into a single embedding token.

    Args:
        metadata_dim : Number of input metadata features M.
        embed_dim    : Output token dimension D (must match CNN embed_dim).
        hidden_dim   : Hidden layer size.
        dropout      : Dropout probability.
    """

    def __init__(
        self,
        metadata_dim: int,
        embed_dim: int = 256,
        hidden_dim: int = 128,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(metadata_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, M)
        Returns:
            token: (B, 1, D)
        """
        return self.net(x).unsqueeze(1)     # (B, 1, D)

### Positional Encoding

We added positional encoding to our input through this class, which simply adds the index position of the time step to the entire sequence vector we get from the CNN and metadata encoder. 

In [5]:
class LearnablePositionalEncoding(nn.Module):
    """
    Adds a learned position embedding to patch tokens.
    The [CLS] and metadata tokens receive position index 0 and N+1
    respectively (handled in the main model by keeping the embedding
    large enough).

    Args:
        max_len  : Maximum sequence length (N_patches + 2 special tokens).
        embed_dim: Token dimension D.
        dropout  : Dropout on the summed representation.
    """

    def __init__(self, max_len: int = 512, embed_dim: int = 256, dropout: float = 0.1):
        super().__init__()
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, L, D)
        Returns:
            x + pos: (B, L, D)
        """
        L = x.size(1)
        positions = torch.arange(L, device=x.device)
        return self.dropout(x + self.pos_embed(positions))

─────────────────────────────────────────────────────────────────────────────
4. Main Model
─────────────────────────────────────────────────────────────────────────────

In [6]:
class ECGArrhythmiaClassifier(nn.Module):
    """
    CNN-Transformer hybrid model for arrhythmia classification from
    ECG spectrograms + patient metadata.

    Architecture summary
    ────────────────────
    ECG spectrogram (B,1,H,W)
        → CNN backbone            (B, N, D)
        → + positional encoding
    Patient metadata (B, M)
        → Metadata MLP            (B, 1, D)
    [CLS] token                   (B, 1, D)

    Concatenated → [CLS | patches₁…ₙ | meta]   (B, N+2, D)
        → Transformer encoder (xL)
        → CLS token               (B, D)
        → Dropout + Linear        (B, num_classes)

    Args:
        num_classes   : Number of arrhythmia categories.
        metadata_dim  : Dimension of patient metadata vector.
        embed_dim     : Shared token embedding dimension D.
        num_heads     : Number of attention heads (must divide embed_dim).
        num_layers    : Number of Transformer encoder layers.
        ffn_dim       : Feed-forward inner dimension in each Transformer layer.
        dropout       : Dropout probability (applied throughout).
        max_seq_len   : Max spectrogram patch count + 2 (for CLS + meta tokens).
        pretrained_cnn: Use ImageNet weights for EfficientNet backbone.
        freeze_cnn_layers: Number of early CNN stages to freeze.
    """

    def __init__(
        self,
        num_classes: int = 4,
        metadata_dim: int = 6,
        embed_dim: int = 256,
        num_heads: int = 8,
        num_layers: int = 4,
        ffn_dim: int = 512,
        dropout: float = 0.1,
        max_seq_len: int = 512,
        pretrained_cnn: bool = True,
        freeze_cnn_layers: int = 4,
    ):
        super().__init__()

        assert embed_dim % num_heads == 0, (
            f"embed_dim ({embed_dim}) must be divisible by num_heads ({num_heads})"
        )

        # ── Submodules ──────────────────────────────────────────────────────
        self.cnn = CNNBackbone(
            embed_dim=embed_dim,
            pretrained=pretrained_cnn,
            freeze_early=(freeze_cnn_layers > 0),
            freeze_layers=freeze_cnn_layers,
        )
        self.meta_encoder = MetadataEncoder(
            metadata_dim=metadata_dim,
            embed_dim=embed_dim,
            hidden_dim=embed_dim // 2,
            dropout=dropout,
        )
        self.pos_encoding = LearnablePositionalEncoding(
            max_len=max_seq_len,
            embed_dim=embed_dim,
            dropout=dropout,
        )

        # Add [CLS] token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,   # expects (B, L, D)
            norm_first=True,    # Pre-LN: more stable training
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(embed_dim),
        )

        # Classification head
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, num_classes),
        )

    # ── Forward ─────────────────────────────────────────────────────────────

    def forward(
        self,
        spectrogram: torch.Tensor,
        metadata: torch.Tensor,
        src_key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            spectrogram          : (B, 1, H, W)   - ECG time-frequency image
            metadata             : (B, M)          - patient metadata vector
            src_key_padding_mask : (B, L) bool     - True = ignore position
                                    (useful if batch contains padding)
        Returns:
            logits               : (B, num_classes)
        """
        B = spectrogram.size(0)

        # 1. Extract CNN patch tokens
        patch_tokens = self.cnn(spectrogram)        # (B, N, D)

        # 2. Encode metadata into a single token
        meta_token = self.meta_encoder(metadata)    # (B, 1, D)

        # 3. Expand [CLS] to batch
        cls_token = self.cls_token.expand(B, -1, -1)  # (B, 1, D)

        # 4. Concatenate: [CLS | patches | meta]
        x = torch.cat([cls_token, patch_tokens, meta_token], dim=1)  # (B, N+2, D)

        # 5. Positional encoding (applied to entire sequence)
        x = self.pos_encoding(x)                    # (B, N+2, D)

        # 6. Transformer encoder
        x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)  # (B, N+2, D)

        # 7. Extract [CLS] token and classify
        cls_out = x[:, 0, :]                        # (B, D)
        logits = self.head(cls_out)                 # (B, num_classes)

        return logits

    # ── Convenience ─────────────────────────────────────────────────────────

    def count_parameters(self) -> dict:
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {"total": total, "trainable": trainable}

### Focal Loss

For our loss function, we implemented focal loss. Focal loss is important because it adds a penalty term that lowers the amount to which confident correct predictions contribute to the loss total. This is important in our case because we don't want the model to keep learning the easy, normal heartbeat predictions. The extra term pushes the loss of a confident, correct prediction towards zero, incentivizing the model to learn the harder, minority class predictions. 

In [7]:
class FocalLoss(nn.Module):
    """
    Focal loss for class-imbalanced arrhythmia datasets.
    Down-weights easy (well-classified) examples so the model focuses
    on hard positives.

    Args:
        gamma    : Focusing parameter (0 = standard cross-entropy).
        weight   : Per-class weights tensor (B,) or None.
        reduction: 'mean' | 'sum' | 'none'
    """

    def __init__(
        self,
        gamma: float = 2.0,
        weight: Optional[torch.Tensor] = None,
        reduction: str = "mean",
    ):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        focal = (1.0 - pt) ** self.gamma * ce
        if self.reduction == "mean":
            return focal.mean()
        elif self.reduction == "sum":
            return focal.sum()
        return focal

### Learning Rate Scheduler

We implemented a learning rate scheduler that was applied to the learning rate for both the CNN backbone as well as the Transformer model. We used a small LR for the pretrained model to keep the gradients from shifting too drastically, while we used a larger LR for the Transformer to have it learn more aggressively. Our scheduler has a small warmup period of 2 epochs to prevent the gradients from growing large too quickly, followed by a cosine decay applied for the rest of training. This schedule prevents gradients from exploding early in training and helps convergence towards the end. 

In [8]:
def build_optimizer_and_scheduler(
    model: ECGArrhythmiaClassifier,
    lr_cnn: float = 1e-4,
    lr_transformer: float = 3e-4,
    weight_decay: float = 1e-4,
    warmup_steps: int = 500,
    total_steps: int = 10000,
):
    """
    Differential learning rates:
      - CNN backbone  → lower LR (pretrained, fine-tune gently)
      - Transformer + head → higher LR (trained from scratch)

    Returns (optimizer, scheduler).
    """
    cnn_params = list(model.cnn.parameters())
    cnn_ids = {id(p) for p in cnn_params}

    other_params = [p for p in model.parameters() if id(p) not in cnn_ids]

    optimizer = torch.optim.AdamW(
        [
            {"params": cnn_params,   "lr": lr_cnn},
            {"params": other_params, "lr": lr_transformer},
        ],
        weight_decay=weight_decay,
    )

    # Linear warmup → cosine decay
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + torch.cos(torch.tensor(3.14159 * progress)).item())

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler

### Test Run

In [ ]:
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Running on: {device}\n")

    #  Config 
    BATCH        = 4
    IMG_H, IMG_W = 224, 224   # spectrogram resolution
    META_DIM     = 6          # [age, sex, n_meds, rr_pre, rr_post, rr_ratio ]
    NUM_CLASSES  = 4          # Normal, AF, VT, SVT

    # Build model 
    model = ECGArrhythmiaClassifier(
        num_classes=NUM_CLASSES,
        metadata_dim=META_DIM,
        embed_dim=256,
        num_heads=8,
        num_layers=4,
        ffn_dim=512,
        dropout=0.1,
        pretrained_cnn=False,   
    ).to(device)

    params = model.count_parameters()
    print(f"Parameters — total: {params['total']:,}  trainable: {params['trainable']:,}\n")

    # Dummy forward pass  
    spectrograms = torch.randn(BATCH, 1, IMG_H, IMG_W).to(device)
    metadata     = torch.randn(BATCH, META_DIM).to(device)
    targets      = torch.randint(0, NUM_CLASSES, (BATCH,)).to(device)

    logits = model(spectrograms, metadata)
    print(f"Input  spectrogram : {tuple(spectrograms.shape)}")
    print(f"Input  metadata    : {tuple(metadata.shape)}")
    print(f"Output logits      : {tuple(logits.shape)}")

    # Loss + backward 
    criterion = FocalLoss(gamma=2.0)
    loss = criterion(logits, targets)
    loss.backward()
    print(f"\nFocal loss         : {loss.item():.4f}")

    optimizer, scheduler = build_optimizer_and_scheduler(model)
    print(f"Optimizer groups   : {len(optimizer.param_groups)}  (CNN + Transformer)")
    print("\nAll checks passed.")

Running on: cpu

Parameters — total: 6,612,227  trainable: 6,303,567



y:\Anaconda\envs\torch\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Input  spectrogram : (4, 1, 224, 224)
Input  metadata    : (4, 6)
Output logits      : (4, 4)

Focal loss         : 0.8199  ✓ backward pass OK
Optimizer groups   : 2  (CNN + Transformer)

All checks passed.
